# PPAT-018: Gene-level fitness effects — Bayesian LMM

**Gain-of-function overexpression screen (Tesseract multi-species library)**

Each barcode carries a genomic fragment expressed in the host.
Fitness = log-ratio of barcode abundance after vs. before selection.

**Baseline** = mean fitness of empty-vector barcodes (`empty_insert`); the neutral null reference.

**Library:** Tesseract (11 donor species). Genes from different species never co-appear in the same
insert, so cross-species gene pairs have r = 0 by construction.
Confounding is expected only between adjacent operonic genes within the same genome.

**Input:** `../data/selection_experiment_insert_data.parquet` (see README's Data section)
**Model:** Pyro SVI Bayesian LMM (`pioneer_hgt_core.bayesian_lmm`)

Ported from PPAT-018/02a_Run_Bayesian_LMM_LB_Salt.ipynb. Writes a single combined dataframe
(`../results/gene_bayesian_lmm_stats.parquet`) rather than the original's full checkpoint
artifact set (those fed a downstream SuSIE finemapping step that isn't part of this repo) --
the diagnostic plots below are kept in full so model convergence/stability remain inspectable.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import pyro
from pathlib import Path
from pyro.infer import Predictive
from scipy.stats import pearsonr

from pioneer_hgt_core.bayesian_lmm import (
    create_sparse_design_matrix,
    scipy_to_torch_sparse,
    fit_lmm_pyro_sparse,
    map_gene_effects_to_names,
    estimate_posterior_p_greater_twosided,
    extract_variance_parameters,
    plot_gene_posterior_comparison,
    get_device,
)

DEVICE = get_device()

# --- data directory (populate yourself -- see README's Data section) ---
DATA_DIR = Path("../data")
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# --- paths ---
INPUT_PARQUET = DATA_DIR / "selection_experiment_insert_data.parquet"

# --- analysis parameters ---
CONDITION = "LB_4_salt"

NUM_ITERATIONS = 1000
LEARNING_RATE  = 0.01
MIN_REPS       = 2        # minimum replicates a barcode must appear in
SEEDS          = [12, 42, 123]

print(f"Device: {DEVICE}")
print(f"Seeds: {SEEDS}  |  iterations: {NUM_ITERATIONS}  |  lr: {LEARNING_RATE}")


## Load data

In [ ]:
raw = pd.read_parquet(INPUT_PARQUET)
print("Shape:", raw.shape)
print()
print("insert_type counts:")
print(raw["insert_type"].value_counts())
print()
print("environment counts:")
print(raw["environment"].value_counts())
print()
print("replicates:", sorted(raw["replicate"].unique()))


## Prepare data

`prepare_condition` mirrors the `prepare_arm` pattern from HAM-003:

1. Estimate baseline from `empty_insert` barcodes (per-barcode fitness mean/std).
2. Keep only `aligned` barcodes present in ≥ `MIN_REPS` replicates.
3. Convert `gene_ids_fully_covered` to plain Python lists (handles numpy arrays / None).
4. Concatenate aligned rows with empty-insert rows (`gene_ids_fully_covered = []`).
   Empty inserts anchor the baseline in the likelihood without contributing gene signal.


In [ ]:
def _to_list(x):
    """Convert gene_ids_fully_covered cell to a plain Python list."""
    if x is None:
        return []
    if isinstance(x, float) and np.isnan(x):
        return []
    if isinstance(x, np.ndarray):
        return x.tolist()
    return list(x)


def prepare_condition(df, environment, min_reps=MIN_REPS):
    """Return (lmm_df, (baseline_mean, baseline_sd)) for a single environment.

    Baseline estimated as mean/std of empty_insert fitness values.
    Aligned barcodes filtered to those present in >= min_reps replicates.
    Empty inserts (gene_ids_fully_covered=[]) are appended to anchor baseline.
    """
    sub = df[df["environment"] == environment].copy()

    empties = sub[sub["insert_type"] == "empty_insert"].copy()
    baseline_mean = float(empties["fitness"].mean())
    baseline_sd   = float(empties["fitness"].std())

    aligned = sub[sub["insert_type"] == "aligned"].copy()
    rep_counts = aligned.groupby("bc_sequence")["replicate"].nunique()
    keep_bc = rep_counts[rep_counts >= min_reps].index
    aligned = aligned[aligned["bc_sequence"].isin(keep_bc)].copy()

    aligned["gene_ids_fully_covered"] = aligned["gene_ids_fully_covered"].apply(_to_list)
    aligned["insert_id"] = aligned["bc_sequence"]

    empties["gene_ids_fully_covered"] = [[] for _ in range(len(empties))]
    empties["insert_id"] = empties["bc_sequence"]

    combined = pd.concat(
        [aligned[["fitness", "gene_ids_fully_covered", "insert_id", "replicate"]],
         empties[["fitness", "gene_ids_fully_covered", "insert_id", "replicate"]]],
        ignore_index=True,
    )

    n_bc    = aligned["bc_sequence"].nunique()
    n_genes = len({g for glist in aligned["gene_ids_fully_covered"] for g in glist})
    n_empty = empties["bc_sequence"].nunique()
    print(f"  {n_bc:,} barcodes | {n_genes:,} unique genes | {n_empty:,} empty barcodes "
          f"| {len(combined):,} total obs")
    print(f"  baseline = {baseline_mean:.4f} \u00b1 {baseline_sd:.4f}  (per-barcode, empty_insert)")
    return combined, (baseline_mean, baseline_sd)


print(f"Preparing condition: {CONDITION}")
lmm_df, (baseline_mean, baseline_sd) = prepare_condition(raw, CONDITION)
lmm_df.head(3)


## Train model

Model per observation:

$$y_{ijk} = \mu + X_i \boldsymbol{\gamma} + u_i^{\text{insert}} + u_k^{\text{rep}} + \varepsilon_{ijk}$$

where $X_i$ is a binary indicator row (which genes barcode $i$ covers), $\boldsymbol{\gamma}$ are the
gene effects of interest, and $u_i^{\text{insert}}$, $u_k^{\text{rep}}$ are random effects.

We warm-start gene effects from the empirical per-gene mean fitness minus baseline, then
run SVI with `guide_type="lowrank"` (default, rank=20). Three independent seeds are used to
check posterior stability.


In [ ]:
# -- Build design matrix --
X_sparse, Y, rep_idx, ins_idx, gene_cols, N_inserts = create_sparse_design_matrix(lmm_df)
N_genes  = len(gene_cols)
N_reps   = int(rep_idx.max()) + 1

print(f"Design matrix: {X_sparse.shape[0]:,} obs x {N_genes:,} genes")
print(f"Replicates: {N_reps}  |  unique inserts: {N_inserts:,}")

X_torch   = scipy_to_torch_sparse(X_sparse)
Y_torch   = torch.tensor(Y, dtype=torch.float32)
rep_torch = torch.tensor(rep_idx, dtype=torch.long)
ins_torch = torch.tensor(ins_idx, dtype=torch.long)

# -- Warm-start: empirical per-gene mean fitness minus baseline --
# Explode multi-gene rows; each gene gets the fitness of every insert it appears in.
gene_mean = (
    lmm_df.explode("gene_ids_fully_covered")
    .dropna(subset=["gene_ids_fully_covered"])
    .query("gene_ids_fully_covered != ''")
    .groupby("gene_ids_fully_covered")["fitness"]
    .mean()
)
init_gene_effects = torch.tensor(
    [float(gene_mean.get(g, baseline_mean)) - baseline_mean for g in gene_cols],
    dtype=torch.float32,
)

n_with_data = (init_gene_effects != 0).sum().item()
print(f"\nInit report:")
print(f"  baseline_mean = {baseline_mean:.4f}  |  baseline_sd = {baseline_sd:.4f}")
print(f"  init_gene_effects  mean={init_gene_effects.mean():.4f}  std={init_gene_effects.std():.4f}  "
      f"min={init_gene_effects.min():.4f}  max={init_gene_effects.max():.4f}")
print(f"  {n_with_data:,} / {N_genes:,} genes have empirical warm-start (rest = 0.0)")

# -- Multi-seed training --
seed_guides, seed_losses = [], []

for seed in SEEDS:
    pyro.clear_param_store()
    pyro.set_rng_seed(seed)
    torch.manual_seed(seed)
    guide, losses = fit_lmm_pyro_sparse(
        X_torch, Y_torch, rep_torch, ins_torch,
        N_genes=N_genes, N_replicates=N_reps, N_inserts=N_inserts,
        baseline_mean=baseline_mean, baseline_std=baseline_sd,
        num_iterations=NUM_ITERATIONS, learning_rate=LEARNING_RATE,
        device=DEVICE, init_gene_effects=init_gene_effects
    )
    seed_guides.append(guide)
    seed_losses.append(losses)
    print(f"  seed={seed}  final ELBO: {losses[-1]:.2f}")

best_idx = min(range(len(seed_losses)), key=lambda i: seed_losses[i][-1])
primary_guide = seed_guides[best_idx]
print(f"\nPrimary guide: seed={SEEDS[best_idx]}  (lowest final ELBO: {seed_losses[best_idx][-1]:.2f})")

## Model checks

### ELBO convergence

All seeds should converge to a similar loss plateau.
A large spread between seeds indicates the posterior is multimodal or learning rate is too high.


In [ ]:
seed_colors = ["steelblue", "tomato", "seagreen"]

fig, ax = plt.subplots(figsize=(8, 3))
for i, losses in enumerate(seed_losses):
    ax.plot(losses, linewidth=0.8, color=seed_colors[i], alpha=0.85, label=f"seed={SEEDS[i]}")
ax.set_xlabel("Iteration")
ax.set_ylabel("ELBO loss")
ax.set_title(f"SVI convergence \u2014 {CONDITION}")
ax.legend()
plt.tight_layout()
fig.savefig(RESULTS_DIR / f"{CONDITION}_elbo_convergence.png", dpi=150)
plt.show()


### Seed stability

Scatter gene effect estimates from seed 0 vs. seeds 1 and 2.
Pearson r > 0.99 confirms the optimizer converges to the same posterior regardless of
initialization. Genes significant in **all** seeds are the most trustworthy hits.


In [ ]:
Q_STABLE = 0.05
fig, axes = plt.subplots(1, len(SEEDS) - 1, figsize=(6 * (len(SEEDS) - 1), 5))
if len(SEEDS) == 2:
    axes = [axes]

effects   = [g.median()["gene_effects"].cpu().numpy() for g in seed_guides]
sig_masks = [
    estimate_posterior_p_greater_twosided(g, gene_cols, N_genes)
    .set_index("gene_name")["q_value"]
    .reindex(gene_cols).values < Q_STABLE
    for g in seed_guides
]
stable_mask = np.all(sig_masks, axis=0)
any_mask    = np.any(sig_masks, axis=0)

for col, j in enumerate(range(1, len(SEEDS))):
    ax = axes[col]
    x, y = effects[0], effects[j]
    r, _ = pearsonr(x, y)

    ax.scatter(x[~any_mask],   y[~any_mask],   s=8,  alpha=0.3, color="lightgray")
    ax.scatter(x[any_mask],    y[any_mask],    s=12, alpha=0.6, color="orange",    label="sig \u22651 seed")
    ax.scatter(x[stable_mask], y[stable_mask], s=16, alpha=0.9, color="steelblue", label="sig all seeds")

    lim = max(abs(x).max(), abs(y).max()) * 1.05
    ax.plot([-lim, lim], [-lim, lim], "k--", linewidth=0.8)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_xlabel(f"seed={SEEDS[0]} gene effect", fontsize=9)
    ax.set_ylabel(f"seed={SEEDS[j]} gene effect", fontsize=9)
    ax.set_title(f"r = {r:.4f}", fontsize=10)
    ax.legend(fontsize=8)

n_stable = stable_mask.sum()
n_any    = any_mask.sum()
print(f"Stable hits (q<{Q_STABLE} in all {len(SEEDS)} seeds): {n_stable} / {n_any} total sig")
plt.suptitle(f"Seed stability \u2014 {CONDITION}", fontsize=10)
plt.tight_layout()
fig.savefig(RESULTS_DIR / f"{CONDITION}_seed_stability.png", dpi=150)
plt.show()


### Variance decomposition

Estimated standard deviations for each random-effect component.


In [ ]:
variance = extract_variance_parameters(primary_guide, N_genes, N_reps, N_inserts)
pd.DataFrame([variance]).T.rename(columns={0: "value"})


## Gene effect estimates

Draw 500 posterior samples from the guide. With `guide_type="lowrank"`, samples propagate
inter-gene correlations; estimated SEs are wider and calibrated for confounded gene pairs
(genes that frequently co-appear in the same insert).

`z_score = mu_gene_effect / sigma_gene_effect`: use |z| > 3 as a hit threshold.


In [ ]:
def extract_gene_effect_samples(guide, X_torch, Y_torch, rep_torch, ins_torch,
                                gene_columns, n_samples=500):
    """Draw posterior samples and return per-gene mean/SE/z-score DataFrame."""
    predictive = Predictive(guide, num_samples=n_samples)
    with torch.no_grad():
        samples = predictive(X_torch, Y_torch, rep_torch, ins_torch)

    gene_samples     = samples["gene_effects"].cpu()           # (n_samples, N_genes)
    baseline_samples = samples["baseline"].cpu().squeeze()     # (n_samples,)

    mu_gene      = gene_samples.mean(0).numpy()
    sigma_gene   = gene_samples.std(0).numpy()
    mu_baseline  = baseline_samples.mean().item()

    return pd.DataFrame({
        "gene_id":        gene_columns,
        "bayes_estimate":   mu_gene,
        "bayes_sigma": np.maximum(sigma_gene, 1e-8),
        "bayes_total_fitness": mu_gene + mu_baseline,
        "bayes_z_score":          mu_gene / np.maximum(sigma_gene, 1e-8),
    }).sort_values("bayes_z_score", key=np.abs, ascending=False).reset_index(drop=True)


gene_effects_df = extract_gene_effect_samples(
    primary_guide, X_torch, Y_torch, rep_torch, ins_torch, gene_cols, n_samples=500
)

gene_effects_df["environment"] = CONDITION

print(f"Top hits by |z-score|:")
print(gene_effects_df.head(15).to_string(index=False))


In [ ]:
# -- Volcano plot: effect size vs. -log10(p-value) --
sig_df = estimate_posterior_p_greater_twosided(primary_guide, gene_cols, N_genes)
sig_df["environment"] = CONDITION
sig_df.rename(columns={"gene_name": "gene_id",
                       "mu_gene_effect": "bayes_estimate",
                       "mu_baseline": "bayes_baseline",
                       "sigma_genes": "bayes_gene_sigma",
                       "mu_total_fitness": "bayes_total_fitness",
                       "sigma_total_fitness": "bayes_total_sigma",
                       "p_value": "bayes_p_value",
                       "q_value": "bayes_q_value"}, inplace=True)
sig_df = sig_df[["gene_id", "environment", "bayes_estimate", "bayes_baseline", "bayes_gene_sigma",
                  "bayes_total_fitness", "bayes_total_sigma", "bayes_p_value", "bayes_q_value"]]
sig_df


In [ ]:
plot_df = gene_effects_df.merge(sig_df[["gene_id", "bayes_p_value", "bayes_q_value"]], on="gene_id")

fig, ax = plt.subplots(figsize=(8, 5))
sig_mask = plot_df["bayes_q_value"] < 0.05
ax.scatter(plot_df.loc[~sig_mask, "bayes_estimate"],
           -np.log10(plot_df.loc[~sig_mask, "bayes_p_value"] + 1e-300),
           s=6, alpha=0.3, color="lightgray", linewidths=0)
ax.scatter(plot_df.loc[sig_mask & (plot_df["bayes_estimate"] > 0), "bayes_estimate"],
           -np.log10(plot_df.loc[sig_mask & (plot_df["bayes_estimate"] > 0), "bayes_p_value"] + 1e-300),
           s=10, alpha=0.7, color="steelblue", linewidths=0, label="sig positive")
ax.scatter(plot_df.loc[sig_mask & (plot_df["bayes_estimate"] < 0), "bayes_estimate"],
           -np.log10(plot_df.loc[sig_mask & (plot_df["bayes_estimate"] < 0), "bayes_p_value"] + 1e-300),
           s=10, alpha=0.7, color="tomato", linewidths=0, label="sig negative")
ax.axvline(0, color="black", linewidth=0.7, linestyle="--")
ax.set_xlabel("Posterior mean gene effect")
ax.set_ylabel("-log10(p-value)")
ax.set_title(f"Volcano plot \u2014 {CONDITION}  (q<0.05 highlighted)")
ax.legend(fontsize=9)
plt.tight_layout()
fig.savefig(RESULTS_DIR / f"{CONDITION}_volcano_plot.png", dpi=150)
plt.show()

n_sig_pos = (sig_mask & (plot_df["bayes_estimate"] > 0)).sum()
n_sig_neg = (sig_mask & (plot_df["bayes_estimate"] < 0)).sum()
print(f"Significant hits (q<0.05): {sig_mask.sum()}  ({n_sig_pos} positive, {n_sig_neg} negative)")


## Additional diagnostics

| Plot | What to look for |
|------|-----------------|
| Z-score histogram | Bulk should track N(0,1); heavy tails = real signal; inflated bulk = over-fitting or unconverged LR |
| Precision plot | Significant genes should have small posterior SD regardless of effect size; upper-right = noisy hits |
| Species breakdown | Disproportionate hits from one species may reflect library coverage bias, not biology |
| Coverage vs. effect | Significant hits should span insert counts; hits only at very low coverage are suspect |
| Effect size histogram | Distribution shape of raw posterior means across sig vs. non-sig genes |


## Save results

Combine `gene_effects_df` (posterior mean/SE/z-score) with `sig_df`'s p/q-values -- reusing the
same merge already built above for the volcano plot (`plot_df`) -- into a single output
dataframe, matching the `bayes_estimate`/`bayes_gene_sigma`/`bayes_z_score`/`bayes_q_value`
column names already used in the public `gene_fitness_results_with_annotations.parquet`.


In [ ]:
gene_bayesian_lmm_stats = plot_df[[
    "gene_id", "environment", "bayes_estimate", "bayes_sigma", "bayes_z_score",
    "bayes_p_value", "bayes_q_value",
]].rename(columns={"bayes_sigma": "bayes_gene_sigma"})

gene_bayesian_lmm_stats.to_parquet(RESULTS_DIR / "gene_bayesian_lmm_stats.parquet", index=False)
out_path = RESULTS_DIR / "gene_bayesian_lmm_stats.parquet"
print(f"Saved {len(gene_bayesian_lmm_stats):,} rows to {out_path}")
gene_bayesian_lmm_stats.head()
